In [1]:
# First generate all possibilities

import itertools as itr

C3_1_prim = [(1,0,0,0),(0,1,0,0),(0,0,1,0),(0,0,0,1)]
C3_2_prim = [(2,0,0,0),(0,2,0,0),(0,0,2,0),(0,0,0,2),\
             (1,1,0,0),(1,0,1,0),(1,0,0,1),(0,1,1,0),(0,1,0,1),(0,0,1,1)]

primaries = []
count_par = 0
count_dup = 0
for poss_prim in itr.product(C3_1_prim,C3_1_prim,C3_2_prim):
    
     # Check if selection rules are satisfied
    par = sum( j*poss_prim[i][j] for i,j in itr.product(range(3),range(1,4)) )
    if par%2:
        count_par+=1
        continue

    # Now check to make sure this primary isn't already in the set of primaries with different labels
    Aposs_prim = tuple(tuple(poss_prim[i][j] for j in range(3,-1,-1)) for i in range(3))
    if Aposs_prim in primaries:
        count_dup+=1
        continue

    # If the primary has passed all checks, add it to the list
    primaries.append(poss_prim)

In [2]:
# Number of primaries with respect to the the subalgebra that commutes with the tdl that exchanges the two C3_1 factors in the numerator
count = 0
for i in range(len(primaries)):
    p = primaries[i]
    if p[0] == p[1]:
        count += 4
    elif (p[1],p[0],p[2]) in primaries[i+1:]:
        continue
    else:
        count += 1
count

72

In [3]:
# Now we wish to calculate the S-matrix

# To do this we need to first calculate the S-matrix of sp(6)_k
# To this end we recall the formula (K is some constant of proportionality):
# S_{\lambda,\mu} = K sum_{w\in W} sgn(w) exp(-\frac{2\pi i}{k+g^{\vee}} <w(\lambda+rho),\mu+\rho)>
# Note that, on the RHS, none of the objects are 'affine'

from sage.groups.matrix_gps.finitely_generated import MatrixGroup

# The rank of the algebra
rank = 3

# The dual Coxeter number
gvee = 4

# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# The Weyl vector
rho = vector([1,1,1])

# The simple roots, coroots, and weights are as follows
simp_roots = [vector([2,-1,0]),vector([-1,2,-1]),vector([0,-2,2])]
simp_lengths = [(root*quadF*root)[0] for root in simp_roots]
simp_coroots = [2*simp_roots[i]/simp_lengths[i] for i in range(rank)]
simp_weights = [vector([1,0,0]),vector([0,1,0]),vector([0,0,1])]

# First let's find the Weyl group (not the affine weyl group)
# There are three simple Weyl reflections

# Calculate the matrix form of the simple Weyl reflections
simp_weyl = []
for i in range(rank):
    root = simp_roots[i]
    coroot = simp_coroots[i]
    # Find the columns of the matrix
    columns = [w - (coroot*quadF*w)[0]*root for w in simp_weights] 
    simp_weyl.append(matrix(columns).T)

# Use these to generate the full Weyl group
weyl_group = MatrixGroup(simp_weyl)
weyl_ref = [matrix(w) for w in list(weyl_group)]

# Now we are prepared to actually calculate the S-matrices of sp(6)_level
kac_C3_1 = [vector(l[1:]) for l in C3_1_prim]
kac_C3_2 = [vector(l[1:]) for l in C3_2_prim]
def S_matrix_sp6(level):
    K = I*8**(-1/2)*(level+gvee)**(-rank/2)
    if level==1:
        kac = kac_C3_1 
    elif level==2:
        kac = kac_C3_2
    S = matrix(SR,len(kac))
    for i,j in itr.product(range(len(kac)),repeat=2):
        S[i,j] = ( K*sum(w.det()*exp(-2*pi*I/(level+gvee)*((w*(kac[i]+rho))*quadF*(kac[j]+rho))[0]) for w in weyl_ref) ).simplify(algorithm='giac')
    
    return S

S1 = S_matrix_sp6(1)
S2 = S_matrix_sp6(2)

# Now we combine these into the S-matrix for the full theory
S_dsp6 = matrix(SR,len(primaries))
for i,j in itr.product(range(len(primaries)),repeat=2):
    pri = primaries[i]
    prj = primaries[j]
    S_dsp6[i,j] = ( 2*S1[C3_1_prim.index(pri[0]),C3_1_prim.index(prj[0])] * S1[C3_1_prim.index(pri[1]),C3_1_prim.index(prj[1])] * S2[C3_2_prim.index(pri[2]),C3_2_prim.index(prj[2])] ).simplify(algorithm='giac')

In [4]:
# Now let's calculate the characters

import itertools as itr

# Affine root system
LC31 = RootSystem(['C', 3, 1]).weight_lattice(extended=True)
# List of fundamental weights
LaC31 = LC31.fundamental_weights()
# The null root delta
deltC31 = LC31.null_root()
# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# Get the affine Weyl group and isolate the classical generators
affine_weyl = LC31.weyl_group()
classical_nodes = [i for i in LC31.index_set() if i != 0]
generators = [affine_weyl.simple_reflection(i) for i in classical_nodes]

# Get the classical Weyl group
classical_weyl = list( RootSystem(['C',3]).weight_lattice().weyl_group() ) # Get the group
classical_weyl = [w.matrix() for w in classical_weyl] # We want a matrix representation

def are_equivalent(w1, w2, level):
    '''Function to determine if two weights are related via shift by a coroot'''
    c1 = w1.monomial_coefficients()
    c2 = w2.monomial_coefficients()
    return all((c1.get(i, 0) - c2.get(i, 0)) % (2 * level) == 0 for i in classical_nodes)

def max_mod_coroots(int_rep, level):
    '''Function that determines the maximal weights in the integrable representation mod the coroots'''
    # Every maximal weight is related to a dominant weight via a classical weyl transform
    dom_wghts = int_rep.dominant_maximal_weights()

    # We will now generate a full set of representations by applying classical weyl transforms
    full_maximal_representatives = list( dom_wghts )
    queue = list( dom_wghts )
    while queue:
        current = queue.pop(0)
        for g in generators:
            nxt = g.action(current)
            # Only keep nxt if it represents a brand new class modulo the coroots
            if not any(are_equivalent(nxt, r, level) for r in full_maximal_representatives):
                full_maximal_representatives.append(nxt)
                queue.append(nxt)
    return full_maximal_representatives


def C3_dcoset_char(lam, mu, nu, max_ord=10):
    '''
    Function to compute the coset character of C3_1xC3_1/C3_2.
    lam,mu,nu should all be triples of nonnegative integers.
    The entries of lam,mu should sum to no more than one, and the entries of nu to no more than two.
    Returns the coset character chi_{lam,mu;nu} of the diagonal coset.
    This is a power series in q with terms up to O(q^{max_ord})
    '''
    if sum(lam) > 1 or sum(mu) > 1 or sum(nu) > 2:
        raise ValueError('Must input affine weights with the correct levels.')

    # Define the rings that the chracters will live in
    P = PuiseuxSeriesRing(ZZ, 'q', sparse = True, default_prec = max_ord+1)
    q = P.gen()

    # Check selection rule before continuing
    if (lam[0]+lam[2]+mu[0]+mu[2]-nu[0]-nu[2])%2 != 0:
        return P(0)

    # The weight lambda in Sagemaths internal weight representation
    Lam = sum(lam[i]*LaC31[i+1] for i in range(3)) + (1-sum(lam))*LaC31[0]
    # This code computes all the string functions necessary for the character
    int_rep = IntegrableRepresentation(Lam)
    max_wghts = max_mod_coroots(int_rep, level=1)
   
    # Compute the strings and modular anomalies
    list_strings = {wght:int_rep.string(wght,depth=max_ord+1) for wght in max_wghts}
    mod_ch = {wght:int_rep.modular_characteristic(wght) for wght in max_wghts}
   
    # String functions in this ring
    strings = {wght: P(list_strings[wght])*q**(mod_ch[wght]) for wght in max_wghts}

    # Convert lam,mu,nu to vectors if they aren't already
    lam = vector(lam)
    mu = vector(mu)
    nu = vector(nu)

    # Define the Weyl vector
    rho = vector([1,1,1])

    char = P(0)
    char = char.add_bigoh(max_ord)
    for Gam in max_wghts:
        this_char = P(0)
        # This is a tuple containing the Dynkin labels
        gam = vector(Gam[i] for i in range(1,4))
        # Loop over the weyl group
        for wr in classical_weyl:
            # The sign of the element is given as its determinant in this representation
            sgn_wr = wr.det()
            for alpha in itr.product(range(-ceil(max_ord/12)*12,(ceil(max_ord/12)+1)*12,12),repeat=3):
                check_vec = wr*(rho+nu) - rho-mu-gam
                if not any(check_vec[i]%2 for i in range(3)):
                    alpha = vector(alpha)
                    evec = wr*(rho+nu) + alpha - (6/5)*(mu+rho)
                    expo = (5/6)*( evec*quadF*evec )[0]/2
                    if expo >=0 and expo <= max_ord:
                        this_char += sgn_wr*q**expo
        # Add this to the result
        char += this_char*strings[Gam] 
    
    # Return the full character
    return char

In [5]:
# Create a list with all the characters

all_dC3_char = list()
for lam,mu,nu in primaries:
    all_dC3_char.append( C3_dcoset_char(lam[1:], mu[1:], nu[1:], max_ord=20) )
    
C3_dcoset_char((0,0,0),(0,0,0),(0,0,0))


q^(-7/120) + q^(233/120) + q^(353/120) + 3*q^(473/120) + 3*q^(593/120) + 7*q^(713/120) + 8*q^(833/120) + 15*q^(953/120) + 19*q^(1073/120) + 30*q^(1193/120) + O(q^10)

In [6]:
c_wghts = list()
for char in all_dC3_char:
    c_wghts.append( char.valuation() + 7/5/24 )

T_dsp6 = diagonal_matrix( [ exp(2*pi*I*(w-7/5/24)) for w in c_wghts ] )


In [7]:
save_data = False
if save_data:
    # Write these for Panos!
    with open('m-dcoset-sp6-S.m','w') as f:
        f.write(mathematica(S_dsp6)._repr_())
    with open('m-dcoset-sp6-T.m','w') as f:
        f.write(mathematica(T_dsp6)._repr_())
    with open('m-dcoset-sp6-conformal-weights.m','w') as f:
        f.write(mathematica(c_wghts)._repr_())

    # Write this guy for me!
    save(S_dsp6,'S-C3-dcoset')

In [8]:
''' Here we calculate additional characters of the invariant algebra (invariant under the interchange of the two C3_1 factors in the numerator) '''

# Affine root system
LC31 = RootSystem(['C', 3, 1]).weight_lattice(extended=True)
# List of fundamental weights
LaC31 = LC31.fundamental_weights()
# The null root delta
deltC31 = LC31.null_root()
# The quadratic form is
quadF = (1/2)*matrix(QQ,[[1,1,1],[1,2,2],[1,2,3]])

# Get the affine Weyl group and isolate the classical generators
affine_weyl = LC31.weyl_group()
classical_nodes = [i for i in LC31.index_set() if i != 0]
generators = [affine_weyl.simple_reflection(i) for i in classical_nodes]

# Get the classical Weyl group
classical_weyl = list( RootSystem(['C',3]).weight_lattice().weyl_group() ) # Get the group
classical_weyl = [w.matrix() for w in classical_weyl] # We want a matrix representation


def C3_char_pm(lam, nu, pm, max_ord = 10):
    ''' 
        Will find the branching functions of the character 
        chi^{pm} = ( chi_lam(q,x)^2 pm chi_lam(q^2,x^2) )/2 into chi_nu.
        It is assumed lam and nu are classical Dynkin labels;
        the zeroeth Dynkin label will be determined assuming lam and nu
        are levels one and two, respectively. pm is plus or minus 1.

        Output: The branching function (character)
    '''

    if sum(lam) > 1 or sum(nu) > 2:
        raise ValueError('Must input affine weights with the correct levels.')

    # Define the rings that the chracters will live in
    P = PuiseuxSeriesRing(ZZ, 'q', sparse = True, default_prec = max_ord+1)
    q = P.gen()

    # The weights in Sagemaths internal weight representation
    Lam = sum(lam[i]*LaC31[i+1] for i in range(3)) + (1-sum(lam))*LaC31[0]
    Nu = sum(nu[i]*LaC31[i+1] for i in range(3)) + (2-sum(nu))*LaC31[0]
    # This code computes all the string functions necessary for the character
    rep_lam = IntegrableRepresentation(Lam)
    rep_nu = IntegrableRepresentation(Nu)
    max_wghts_lam = max_mod_coroots(rep_lam, level=1)
    max_wghts_nu = max_mod_coroots(rep_nu, level=2)

    all_lvl2_wghts = [ sum(wght[i]*LaC31[i] for i in range(4)) for wght in C3_2_prim ]
    all_lvl2_reps = { wght: IntegrableRepresentation(wght) for wght in all_lvl2_wghts}
    all_lvl2_max = { wght: [wa.to_classical() for wa in max_mod_coroots( all_lvl2_reps[wght], level=2 )] for wght in all_lvl2_wghts }
    all_lvl2_max_wghts = list( set( wght for wghts in all_lvl2_max.values() for wght in wghts ) )
    
    # Build all the equations given the choice of lambda
    # Dictionary indexed by eta (the argument of theta)
    # Each value is a pair of lists
    # The first list is string functions sigma^(lam)_mu (here 2*mu=eta)
    # The second is string functions sigma^(nu)_eta
    eqns = [] 
    all_var = []
    var_count = 0
    branch_func_count = 0
    all_bf = []
    all_bf_var = []
    bvec = vector(SR,len(all_lvl2_max_wghts))
    mat = matrix(SR,len(all_lvl2_max_wghts),len(all_lvl2_wghts))
    for i in range(len(all_lvl2_max_wghts)):
        eta = all_lvl2_max_wghts[i]
        lhs = 0
        for mu in max_wghts_lam:
            if 2*mu.to_classical() == eta:
                lhs = var(f'x_{var_count}')
                all_var.append((lam,mu))
                var_count += 1
        bvec[i] = lhs
        rhs = 0
        for j in range(len(all_lvl2_wghts)):
            nu = all_lvl2_wghts[j]
            this_rhs = 0
            if eta in all_lvl2_max[nu]:
                this_rhs += var(f'x_{var_count}')
                all_var.append((nu,eta))
                var_count += 1
            if nu in all_bf:
                ind = all_bf.index(nu)
            else:
                ind = branch_func_count
                all_bf_var.append(var(f'y_{ind}'))
                branch_func_count += 1
                all_bf.append(nu)
            mat[i,ind] = this_rhs
            rhs += this_rhs*var(f'y_{ind}')
        
        eqns.append(lhs == rhs)
    print(mat*vector(SR,all_bf_var))
    print(eqns)
    print(len(eqns))
    print(all_bf_var)
    print(mat.solve_right(bvec))
    #print(solve(eqns,all_bf_var,algorithm='giac'))
    return
    all_nu_on_rhs = dict() # Dictionary to store all nu's and eta's that can appear on the rhs given the lambda
    for poss_nu in all_lvl2_wghts:
        all_eta = [] 
        for poss_eta in all_lvl2_max[poss_nu]:
            for poss_mu in max_wghts_lam:
                if poss_eta == 2*poss_mu:
                    all_eta.append(poss_eta)
                    break
        if all_eta:
            all_nu_on_rhs[poss_nu] = all_eta
    
    all_nu_eta_on_rhs = {mu: [] for mu in max_wghts_lam}
    for poss_mu in max_wghts_lam:
        for poss_nu in all_lvl2_wghts:
            if poss_eta in all_lvl2_max[poss_nu]:
                all_nu_eta_on_rhs[mu].append((poss_nu,))

    # String functions on RHS
    all_needed_nu_strings = { (wght1,wght2): \
                        P( all_lvl2_reps[wght1].string(wght2,depth=max_ord+1) )*q**(all_lvl2_reps[wght1].modular_characteristic(wght2)) \
                        for wght1 in all_nu_on_rhs for wght2 in all_nu_on_rhs[wght1]  }
   
    # String functions on LHS
    strings_2lam = {wght: P([2*ex for ex in rep_lam.string(wght,depth=max_ord+1)])*q**(2*rep_lam.modular_characteristic(wght)) for wght in max_wghts_lam}

    # Now we build the linear system of equations
    var_count = 0
    all_lhs_vars = dict()
    for mu in max_wghts_lam:
        var_count += 1
        all_lhs_vars[mu] = var(f'x_{var_count}')
    all_rhs_vars = dict()
C3_char_pm((0,0,0),(0,0,0),1,max_ord=5)

(x_0*y_0 + x_1*y_1 + x_2*y_2 + x_3*y_3 + x_4*y_5 + x_5*y_8, x_6*y_0 + x_7*y_1 + x_8*y_2 + x_9*y_3 + x_10*y_5 + x_11*y_8, x_12*y_0 + x_13*y_1 + x_14*y_2 + x_15*y_3 + x_16*y_5 + x_17*y_8, x_18*y_4 + x_19*y_6 + x_20*y_7 + x_21*y_9, x_22*y_4 + x_23*y_6 + x_24*y_7 + x_25*y_9, x_26*y_0 + x_27*y_1 + x_28*y_2 + x_29*y_3 + x_30*y_5 + x_31*y_8, x_32*y_0 + x_33*y_1 + x_34*y_2 + x_35*y_3 + x_36*y_5 + x_37*y_8, x_38*y_0 + x_39*y_1 + x_40*y_2 + x_41*y_3 + x_42*y_5 + x_43*y_8, x_44*y_4 + x_45*y_6 + x_46*y_7 + x_47*y_9, x_48*y_4 + x_49*y_6 + x_50*y_7 + x_51*y_9, x_52*y_0 + x_53*y_1 + x_54*y_2 + x_55*y_3 + x_56*y_5 + x_57*y_8, x_58*y_4 + x_59*y_6 + x_60*y_7 + x_61*y_9, x_62*y_4 + x_63*y_6 + x_64*y_7 + x_65*y_9, x_66*y_4 + x_67*y_6 + x_68*y_7 + x_69*y_9, x_70*y_4 + x_71*y_6 + x_72*y_7 + x_73*y_9, x_74*y_4 + x_75*y_6 + x_76*y_7 + x_77*y_9, x_78*y_0 + x_79*y_1 + x_80*y_2 + x_81*y_3 + x_82*y_5 + x_83*y_8, x_84*y_4 + x_85*y_6 + x_86*y_7 + x_87*y_9, x_88*y_0 + x_89*y_1 + x_90*y_2 + x_91*y_3 + x_92*y_5 + x_93

ValueError: matrix equation has no solutions

In [ ]:
# Now we calculate the T-matrices of C3 levels one and two
# First we calculate the modular anomalies and T-matrix of sp(6)_1 and sp(6)_2
moda1 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(1+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_1 ]
moda2 = [ ((l+rho)*quadF*(l+rho))[0]/(2*(2+gvee)) - ((rho)*quadF*(rho))[0]/(2*gvee) for l in kac_C3_2 ]
T1 = diagonal_matrix( [ exp( 2*pi*I*m ) for m in moda1 ] )
T2 = diagonal_matrix( [ exp( 2*pi*I*m ) for m in moda2 ] )

In [ ]:
# Here we deinf a function to bootstrap all possible modular invariants
def sym_commutant_basis(matrix_list):
    """
    Return a basis for the commutant of the matrices in matrix_list
    as a list of matrices.
    """
    if not isinstance(matrix_list,list):
        matrix_list = [matrix_list]
    
    # It is assumed thse are the same for all matrices in the list
    A0 = matrix_list[0]
    K = A0.base_ring()
    n = A0.nrows()

    Id = identity_matrix(K, n)
    Zero = matrix(K, n)
    # Start by enforcing symmetry
    M = identity_matrix(ZZ, n^2) - sum( matrix(ZZ, n, {(i,j):1}).tensor_product(matrix(ZZ, n, {(j,i):1})) for i,j in itr.product(range(n),repeat=2) )

    for A in matrix_list:
        # Build a matrix of linear constraints on the matrix entries
        M = M.stack(Id.tensor_product(A.transpose()) - A.tensor_product(Id))

    # Solve the constraints
    ker = M.right_kernel()

    # Convert vectors back into matrices
    basis = []
    for v in ker.basis():
        X = matrix(K, n, n, list(v))
        basis.append(X)

    return basis

In [ ]:
# Find all the modular invariants of C3 level 2 (at level 1 there is only the diagonal)
# The spacific choices of a and b were determined by non-negativity and integrality
var('a','b')

mod_basis = sym_commutant_basis([matrix(QQbar,S2),matrix(QQbar,T2)])

for M in mod_basis:
    for i,j in itr.product(range(M.nrows()),repeat=2):
        M[i,j] = M[i,j].radical_expression()

pM = mod_basis[0] + a*mod_basis[1] + b*mod_basis[2]
all_mod_inv = [pM.subs(a=0).subs(b=1),pM.subs(a=1).subs(b=0),pM.subs(a=0).subs(b=0)]
all_mod_inv

[
[1 0 0 0 0 0 0 0 0 0]  [1 0 0 0 0 0 0 0 1 0]  [1 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0 0 0]  [0 0 0 0 0 0 0 0 0 0]  [0 1 0 0 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0 0 0]  [0 0 0 0 0 0 0 0 0 0]  [0 0 1 0 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]  [0 0 0 1 0 1 0 0 0 0]  [0 0 0 1 0 0 0 0 0 0]
[0 0 0 0 1 0 0 0 0 0]  [0 0 0 0 0 0 0 0 0 0]  [0 0 0 0 0 0 0 0 0 1]
[0 0 0 0 0 1 0 0 0 0]  [0 0 0 1 0 1 0 0 0 0]  [0 0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 0 1 0 0 0]  [0 0 0 0 0 0 0 0 0 0]  [0 0 0 0 0 0 1 0 0 0]
[0 0 0 0 0 0 0 1 0 0]  [0 0 0 0 0 0 0 2 0 0]  [0 0 0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 0 0 1 0]  [1 0 0 0 0 0 0 0 1 0]  [0 0 0 0 0 0 0 0 1 0]
[0 0 0 0 0 0 0 0 0 1], [0 0 0 0 0 0 0 0 0 0], [0 0 0 0 1 0 0 0 0 0]
]

In [ ]:
%%cython

'''
This code will find the combination of characters to produce the characters of the maximal chiral algebra for
Non-diagonal M_dcoset. In other words, we want A such that M_dcoset = A^tA
'''

from sage.combinat.permutation import Permutations
from sage.functions.other import ceil
from sage.matrix.constructor import matrix
from sage.matrix.special import identity_matrix
import itertools

#import os
#os.environ["SAGE_NUM_THREADS"] = '8'


# Function to compute the integer square root
# Both math.isqrt from python and isqrt from SageMath are 2-10 times slower
cdef int isqrt(int n) noexcept nogil:
    cdef int x = n
    cdef int y = (x + 1) // 2
    while y < x:
        x = y
        y = (x + n // x) // 2
    return x

# Function that recursively computes the decompositions of n
# The `start` index ensures that the squares are chosen in non-increasing order
# This could be viewed as a helper function for a function that took in only n
# Should have squares = [i*i for i in range(max_i+1)], max_i = isqrt(n)t l
cdef list sum_of_squares(int n, list squares, int max_i):
    # Base case: exactly decomposed n.
    if n == 0:
        return [[]]
    
    cdef list res = [] 
    cdef int i, sq
    
    # Try each square starting from index `start`
    for i from max_i >= i >= 1 by 1:
        sq = squares[i]
        if sq <= n:
            # For each decomposition of the remainder, prepend the current square.
            for sub in sum_of_squares(n - sq, squares, i):
                res.append([ i ] + sub)
    
    return res

cdef inline list subtract_from_Mm(list Mm, int s, list P, int j):
    cdef Py_ssize_t i
    cdef int new
    cdef list ret = []
    for i in range(len(Mm)):
        new = (<Py_ssize_t> Mm[i]) - s*(<Py_ssize_t> P[i][j])
        if new >= 0:
            ret.append(new)
        else:
            return []
    return ret


cdef list split_S( list Mm, list S, list P, int start_S, list remaining):
    
    # Base case: exactly decomposed Mm.
    if Mm == [0]*len(Mm):
        return [(S.copy(),[0]*len(P[0]))]# Algorithm to compute the characters of an affine lie algebra

    
    cdef list Mm_minus, result = [] 
    cdef int i,j,s,m,r
    # Try each element of S,P starting with S from index `start_S`
    for i in range(start_S, len(S)):
        for j in range(len(remaining)):
            s = S[i]
            r = remaining[j]
            Mm_minus = subtract_from_Mm(Mm, s, P, r)
            if Mm_minus:
                # For each decomposition of the remainder, adjust the output
                for sub in split_S(Mm_minus, S, P, i+1, remaining[:j]+remaining[j+1:]):
                    sub[0].pop(i)
                    sub[1][r] = s
                    #if len(result) == 0 or result[len(result)-1] != sub:
                    if sub not in result:
                        result.append(sub)
    
    return result


# We find all the partial-m decompositions of a given matrix M
# M is a symmetric square matrix of non-negative integers
# This is the iterative step over m in Algorithm 5.13 of the reference
cdef list Partial_Decomp(list all_P, M, int m_max):
    
    # Initialise some variables
    cdef int k, s, v, l, p, m, ind_P, max_i, M_diag
    cdef list S, Sc, P, V

    for ind_P from len(all_P)-1 >= ind_P >= 0 by 1:
        # Remove the considered matrix from all_P
        # Notice we work in order of decreasing indices so this will not affect future iterations of the loop
        P = all_P.pop(ind_P)
        
        # This will be used a bunch
        m = len(P)
        
        # Check to see whether the matrix has no rows (m=0), or has terminated for this P
        # This check ensures the algorithm has not terminated for this P
        if m == m_max:
            # If this is the case we return the matrix to the list
            # We also cast it as a SageMath matrix
            all_P += [ matrix(P) ]
            continue
        
        # The m=0 case must be dealt with seperatley as m=k=0 means P = [] so P[0] is out of range
        elif m == 0: 
            k = 0
            
        # This is the generic non-terminated case
        else:
            k = len(P[0]) # P is an m by k matrix
            
        # Define this diagonal element
        M_diag = M[m,m]
        
        if k==0: # This means P is empty
            # Case 1
            if M_diag == 0:
                all_P += Partial_Decomp([ P + [[]] ], M, m_max)
            # Case 2
            else:
                # This finds a sum of squares decomposition
                # It will be given as [p_1,..,p_l] where M_diag = p_1+...+p_|l
                # and each p_l is a perfect square
                max_i = isqrt(M_diag)
                for S in sum_of_squares(M_diag, [i * i for i in range(max_i + 1)], max_i):
                    all_P += Partial_Decomp([ [[0]*len(S)]*m + [S] ], M, m_max)
                    
        else: # This means P is non-empty
            # Case 3
            if M_diag == 0:
                all_P += Partial_Decomp([ P + [[0]*len(P[0])] ], M, m_max)
            # Case 4
            else:
                # This finds a sum of squares decomposition
                # It will be given as [p_1,..,p_l] where M[m,m] = p_1**(1/2)+...+p_l**(1/2)
                # and each p_l is an integer
                max_i = isqrt(M_diag)
                for S in sum_of_squares(M_diag, [i * i for i in range(max_i + 1)], max_i):
                    for Sc,V in split_S([M[l,m] for l in range(m)], S, P, 0, list(range(k))):
                        all_P += Partial_Decomp([ [ P[l]+[0]*len(Sc) for l in range(m) ] + [ V + Sc ] ], M, m_max)
                    
    return all_P

def Decomps(M):
    '''
    Function to decompose the integer marix M into an integer matrix A such that M = A*A^t
    Will find all such A and return them as a list.
    '''
    M = matrix(M)
    return Partial_Decomp([[]], M, M.nrows())

In [ ]:
all_max_char = []
all_char_decomp = []
M_C3_1 = identity_matrix(4) # The diagonal modular matrix of the level one thoery is the only one
for mod_inv in all_mod_inv:
    M_dc = matrix(ZZ, 40)
    for i,j in itr.product(range(40),repeat=2):
        pri = primaries[i]
        prj = primaries[j]
        M_dc[i,j] = product( M_C3_1[C3_1_prim.index(pri[k]),C3_1_prim.index(prj[k])] for k in [0,1] ) * mod_inv[C3_2_prim.index(pri[2]),C3_2_prim.index(prj[2])]

    max_chiral_decomp = Decomps(M_dc)[0] # There's only a single decomposition for each
    all_char_decomp.append(max_chiral_decomp)
    all_max_char.append( [ sum(all_dC3_char[i]*max_chiral_decomp[i,j] for i in range(max_chiral_decomp.nrows())) for j in range(max_chiral_decomp.ncols()) ] )

NameError: name 'all_mod_inv' is not defined

In [ ]:
all_char_decomp

[40 x 40 dense matrix over Integer Ring,
 40 x 16 dense matrix over Integer Ring,
 40 x 32 dense matrix over Integer Ring]

In [ ]:
''' Now we attempt to identify the gauging that porduces the three choices of modular invariant '''
nim_irreps = load('NIM-irreps-C3_2')

def N_from_S(S):
    num_L = S.ncols()
    S = matrix(QQbar, S)
    Sn = matrix(CC,S)
    N = [[]]*num_L
    qdim = [[]]*num_L
    for i in range(num_L):
        qdim[i] = ( S[i,0] / S[0,0] ).radical_expression()
        N[i] = [ [ZZ((sum( Sn[i,m]*Sn[j,m]*conjugate(Sn[l,m])/Sn[0,m] for m in range(num_L) )).real().round()) \
                  for l in range(num_L)] for j in range(num_L) ]

    return N, qdim

N2, qdim = N_from_S(S2)

# Let us now search for binary algebras as these will be easiest to gauge
bi_algs = []
for irrep in nim_irreps:
    algebras = irrep[0]
    bi_algs.append([alg for alg in algebras if sum(alg)==2])
bi_algs

[[],
 [(1, 0, 1, 0, 0, 0, 0, 0, 0, 0), (1, 0, 0, 1, 0, 0, 0, 0, 0, 0)],
 [],
 [],
 [],
 [(1, 0, 0, 0, 0, 0, 0, 0, 1, 0)],
 [],
 [],
 []]

In [ ]:
# Consistency check
test_rep = nim_irreps[2][1]
for i,j in itr.product(range(10),repeat=2):
    if test_rep[i]*test_rep[j] != sum(N2[i][j][k]*test_rep[k] for k in range(10)):
        print(i,j)

In [ ]:
# Numerical tolerance on computations (only relevant if our_field=CC)
tol = 1e-8

# The field we will work over
# The two choices are CC and QQbar; CC is approximate, but faster, QQbar is exact, but slower
our_field = CC

# Number of primaries
num_prim = len(C3_2_prim)

# First define the defects involved in the gauging
indL = 2
L = C3_2_prim[indL]
e = C3_2_prim[0]
# Now define the algebra
Alg = [0, indL]
dimA = QQbar(sum(S2[ind,0] / S2[0,0] for ind in Alg))

## Let's define the components we know here
space_M = diagonal_matrix( SR,[ mathematica(S2[i,indL]/S2[i,0]).FullSimplify().sage() for i in range(num_prim) ] )
time_M = matrix( mathematica(S2.H*space_M*S2).FullSimplify().sage() )
both_M = matrix( mathematica(T2.H*time_M*T2).FullSimplify().sage() )
Stran_both_M = matrix( mathematica(S2.H*both_M*S2).FullSimplify().sage() )

# The primaries that enter in the expansion of the twisted Hilbert space partition function
var_ent = []
for p in itr.product(range(num_prim),repeat=2):
    if time_M[p]!=0:
        var_ent.append(p)
# Total number of variables needed to solve for the action of L on the primaries
num_var = len(var_ent)

# Generate the ansatz for the remaining diagram (the one that's not the identity, space_M, time_M, or both_M)
Npoly = PolynomialRing(our_field,num_var,'n')
Ngens = Npoly.gens()
ansatz = matrix(Npoly,num_prim,num_prim)
gen_num = 0
for p in var_ent:
    ansatz[p] = time_M[p]*Ngens[gen_num]
    gen_num += 1

# Get the constraints as entries of the matrices eqn_matrix1 and eqn_matrix2
nS2 = matrix(Npoly,S2)
nT2 = matrix(Npoly,T2)
Herm_nT2 = matrix(Npoly,T2.H)
Stran_ansatz = nS2*ansatz*nS2
eqn_matrix1 = -ansatz + Stran_ansatz - matrix(Npoly,both_M) + matrix(Npoly,Stran_both_M)
eqn_matrix2 = Herm_nT2*Stran_ansatz*nT2 - ansatz

## Organise the constraints into a linear system
constraint_matrix = []
bvec = []
for p in itr.product(range(num_prim),repeat=2):
    row = [eqn_matrix1[p][g] for g in Ngens]
    vec_ent = eqn_matrix1[p]([0]*num_var)
    if any(row):
        constraint_matrix.append(row)
        bvec.append(vec_ent)
    elif abs(vec_ent) > 1e-6:
        print('uh-oh',p)
    row = [eqn_matrix2[p][g] for g in Ngens]
    vec_ent = eqn_matrix2[p]([0]*num_var)
    if any(row):
        constraint_matrix.append(row)
        bvec.append(vec_ent)
    elif abs(vec_ent) > 1e-6:
        print('uh-oh',p)
constraint_matrix = matrix(our_field,constraint_matrix)
bvec = vector(our_field,bvec)

# Our system appears overconstrained, but we know that an exact solution must exist as the gauging is non-anomolous
# We use linear least squares to find the solution as it is much more numerically stable than .solve_right
# This works as linear least squares is guarenteed to find the exact solution if it exists, and we know it exists
print(constraint_matrix.right_kernel()) # check to make sure we find the unique solution
LLSm = constraint_matrix.T*constraint_matrix
our_sol = LLSm.inverse()*constraint_matrix.T*bvec

# Actually build the final missing matrix
both_Mpp = matrix(our_field,num_prim,num_prim)
ent = 0
for p in var_ent:
    both_Mpp[p] = our_sol[ent]*time_M[p]
    ent+=1

mult_map = 1/sqrt( dimA )
# Compute the gauged modular matrix
M_gauged = mult_map**2*identity_matrix(our_field, num_prim) + mult_map**2*space_M + mult_map**2*time_M + mult_map**2*both_M - mult_map**2*both_Mpp
# If we have computed M_gauged numerically, we round to integers within numerical tolerance
if our_field==CC:
    for p in itr.product(range(num_prim),repeat=2):
        nt = CC(M_gauged[p])
        if abs(imag(nt)) > tol:
            print(p,'Imaginary')
        elif real(nt) < -tol:
            print(p,'negative')
        elif abs( real(nt) % 1 ) > tol:
            print(p,'non-integral')
        else:
            M_gauged[p] = round(real(nt))
try:
    M_gauged = matrix(ZZ,M_gauged)
    if M_gauged in all_mod_inv:
        print('Gauging!')
    else:
        raise
except:
    print('Check failed')
    var_ent.append(p)
# Total number of variables needed to solve for the action of L on the primaries
num_var = len(var_ent)


Vector space of degree 24 and dimension 0 over Complex Field with 53 bits of precision
Basis matrix:
[]
Gauging!


In [45]:
M_gauged

[1 0 0 0 0 0 0 0 0 0]
[0 1 0 0 0 0 0 0 0 0]
[0 0 1 0 0 0 0 0 0 0]
[0 0 0 1 0 0 0 0 0 0]
[0 0 0 0 0 0 0 0 0 1]
[0 0 0 0 0 1 0 0 0 0]
[0 0 0 0 0 0 1 0 0 0]
[0 0 0 0 0 0 0 1 0 0]
[0 0 0 0 0 0 0 0 1 0]
[0 0 0 0 1 0 0 0 0 0]

In [29]:
our_sol

(0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000)

In [30]:
constraint_matrix.right_kernel()

Vector space of degree 10 and dimension 0 over Complex Field with 53 bits of precision
Basis matrix:
[]

In [31]:
constraint_matrix.solve_right(bvec)

(0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000, 0.000000000000000)

In [20]:
%%cython
''' Code to decompose the characters we have found in terms of the characters of fixed point algebra'''
from sage.rings.puiseux_series_ring import PuiseuxSeriesRing
from sage.rings.rational_field import QQ
from sage.rings.integer_ring import ZZ

from sage.matrix.constructor import matrix
from sage.modules.free_module_element import vector

from sage.misc.functional import isqrt

from sage.combinat.partition import number_of_partitions
import itertools as itr



def char_decomp(list char1, list char2, int max_ord = 10):
    '''
    Function to compute the decomposition of the characters in char1 into the characters in char2.
    max_ord is the order up to which the decomposition is checked
    A list of lists called solutions is returned.
    The number solutions[i][j] is the number of times character j in char2 appears in the decomposition of character i in char1
    '''

    # Power Series Ring the characters live in
    P = PuiseuxSeriesRing(QQ, 'q', sparse=True, default_prec=max_ord)

    # To speed things up, let us first check which decompositions are in principle possible 
    # according to the fractional part of the character exponents
    cdef list poss_decomps = [[] for _ in range(len(char1))]
    cdef int i,j
    cdef list val1 = [ QQ( char1[i].valuation() ) for i in range(len(char1)) ]
    cdef list val2 = [ QQ( char2[i].valuation() ) for i in range(len(char2)) ]
    for i,j in itr.product(range(len(char1)), range(len(char2))):
        if (val1[i]-val2[j]).denominator() == 1:
            poss_decomps[i] += [j]
    print(poss_decomps)
            
    # Now we actually compute the decompositions
    cdef list expo
    cdef int num_char,max_ord_eff,n
    cdef int l,k
    # List of solutions
    cdef list solutions = [[] for _ in range(len(char1))]
    max_num = min([len(char.exponents())-1 for char in char2] + [len(char.exponents())-1 for char in char1])
    print(max_num)
    for i in range(len(char1)):
        # To save time just skip ones where no decomposition is possible.
        if not poss_decomps[i]:
            continue
        # We now try solving it up to order 'n' and increase n until we fix a unique solution
        num_char = len(poss_decomps[i])
        char = char1[i]
        for n in range(num_char, max_num):
            expo = char.exponents()[:n]
            Vec = vector(ZZ, n)
            Mat = matrix(ZZ, num_char, n)
            for l in range(n):
                Vec[l] = char[expo[l]]
                for k in range(num_char):
                    Mat[k,l] = char2[poss_decomps[i][k]][expo[l]]
            # If the equation has no solution a ValueError is raised and we continue
            try:
                this_sol = Mat.solve_left(Vec)
            except ValueError:
                continue
            this_kern = Mat.kernel().basis()
            
            if len(this_kern)==0 or n == max_num-1:
                com_char = sum(P(this_sol[j]*char2[poss_decomps[i][j]], prec = max_ord) for j in range(num_char))
                if P(com_char-char, prec = max_ord-1) == 0:
                    solutions[i] += [ this_sol, poss_decomps[i] ]
                break

    # Return all the decompositions we have found
    return solutions


In [21]:
# Load list of various characters
inv_char = load('invariant-algebra-characters')
inv_index = list( inv_char.keys() )
inv_char_list = list( inv_char.values() )
trisq_char = load('tricritical-ising-squared-characters')

all_decomps = []
for max_char in all_max_char:
    all_decomps.append( char_decomp(max_char,inv_char_list) )

[[0, 1, 15, 16], [], [], [19], [19], [0, 1, 15, 16], [], [], [38], [], [13, 20], [], [], [8, 21], [8, 21], [13, 20], [], [], [30, 31], [], [], [], [38], [], [14], [], [], [4, 5, 9, 10], [4, 5, 9, 10], [14], [], [], [22, 23], [], [8, 21], [], [], [13, 20], [13, 20], [8, 21]]
16
[[0, 1, 15, 16], [19], [38], [38], [13, 20], [8, 21], [30, 31], [30, 31], [38], [38], [14], [4, 5, 9, 10], [22, 23], [22, 23], [8, 21], [13, 20]]
18
[[0, 1, 15, 16], [], [], [19], [19], [0, 1, 15, 16], [], [38], [13, 20], [], [], [8, 21], [8, 21], [13, 20], [], [30, 31], [], [38], [14], [], [], [4, 5, 9, 10], [4, 5, 9, 10], [14], [], [22, 23], [8, 21], [], [], [13, 20], [13, 20], [8, 21]]
16


In [22]:
all_decomps[1]

[[(1, 1, 1, 1), [0, 1, 15, 16]],
 [(2), [19]],
 [(1), [38]],
 [(1), [38]],
 [(1, 1), [13, 20]],
 [(1, 1), [8, 21]],
 [(1, 1), [30, 31]],
 [(1, 1), [30, 31]],
 [(1), [38]],
 [(1), [38]],
 [(2), [14]],
 [(1, 1, 1, 1), [4, 5, 9, 10]],
 [(1, 1), [22, 23]],
 [(1, 1), [22, 23]],
 [(1, 1), [8, 21]],
 [(1, 1), [13, 20]]]

In [45]:
dA1_char = load('su2-dcoset-chars')
char_decomp(dA1_char,all_dC3_char)

[[0, 5], [2], [3, 4], [0, 5], [1], [3, 4], [17], [18], [16, 19], [18], [17], [3, 4], [1], [0, 5], [3, 4], [2], [0, 5], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [27, 28], [25], [24, 29], [27, 28], [26], [24, 29], [31], [32], [30, 33], [32], [31], [24, 29], [26], [27, 28], [24, 29], [25], [27, 28], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [], [10, 15, 37, 38], [12, 35], [13, 14, 34, 39], [10, 15, 37, 38], [11, 36], [13, 14, 34, 39], [7, 21], [8, 22], [6, 9, 20, 23], [6, 9, 20, 23]]
4


[[],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [(1, 0), [16, 19]],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [(1, 0), [30, 33]],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [(1, 0), [12, 35]],
 [],
 [],
 [(1, 0), [11, 36]],
 [],
 [(1, 0), [7, 21]],
 [(1, 0), [8, 22]],
 [],
 []]

In [ ]:
from itertools import product

def delinge(N1, N2):
    '''Function that takes in two fusion rules N1, N2 and outputs the fusion rules of the Delinge tensor product.'''

    ind = list( product(range(len(N1)), range(len(N2))) )
    N = [ [ [ 0 for _ in range(len(ind)) ] for _ in range(len(ind)) ] for _ in range(len(ind)) ]
    for i,j,k in product(range(len(ind)), repeat=3):
        N[i][j][k] = N1[ind[i][0]][ind[j][0]][ind[k][0]] * N2[ind[i][1]][ind[j][1]][ind[k][1]]
    return N, ind

def semidirect_exchange(No, qdimo):
    '''Function that takes in fusion rules No and outputs the fusion ruls of the Delinge tensor product of No with itself
    with an additional semidirect product with a Z_2 that exchanges the two copies of No.'''

    ind = list( product(range(len(No)), range(len(No)), range(2)) )
    N = [ [ [ 0 for _ in range(len(ind)) ] for _ in range(len(ind)) ] for _ in range(len(ind)) ]
    qdim = [ qdimo[i]*qdimo[j] for i,j,_ in ind ]
    for i,j,k in product(range(len(ind)), repeat=3):
        if (ind[i][2] + ind[j][2] + ind[k][2])%2 == 1:
            N[i][j][k] = 0
        elif ind[j][2] == 0:
            N[i][j][k] = No[ind[i][0]][ind[j][0]][ind[k][0]] * No[ind[i][1]][ind[j][1]][ind[k][1]]
        else:
            N[i][j][k] = No[ind[i][0]][ind[j][1]][ind[k][1]] * No[ind[i][1]][ind[j][0]][ind[k][0]]

    return N, ind, qdim

In [ ]:
'''Now we find all modular matrices that can be obtained from gauging by an algebra object'''

# Let's first get a list of group-like symmetries

N, qdim = N_from_S(S_dsp6)

gr_ind = [i for i in range(len(qdim)) if qdim[i]==1]
# There is just a Z_2 group; nothing more
z2_ind = gr_ind[1]
# Let's get the action of this tdl on the primaries
Tc = matrix(CC,T_dsp6,sparse=True)
Sc = matrix(CC,S_dsp6)
tdl = vector(CC, [ Sc[z2_ind,i]/Sc[0,i] for i in range(len(primaries)) ] )
# Now let's get the modular invariant when we gauge by this line
M_diag = identity_matrix(CC, len(primaries), sparse=True)
M_space = diagonal_matrix(CC, tdl, sparse=True)
M_time = Sc*M_space*Sc
M_both = Tc.H*M_time*Tc
M_gauged_c = (M_diag+M_space+M_time+M_both)/2

# Let's round to get the integer valued matrix
M_gauged = matrix(ZZ,len(primaries))
for i,j in itr.product(range(len(primaries)), repeat=2):
    M_gauged[i,j] = round( M_gauged_c[i,j].real(), ndigits=0 )


In [7]:
print( (T2*T2.H-identity_matrix(10)).n().norm() )
print( ((S2.n()*T2.H.n())**3 - S2.n()**2).n().norm() )
print( S2.is_symmetric() )
print( (S2*S2.H-identity_matrix(10)).n().norm() )

0.0
5.470990389089858e-16
True
1.856963405217746e-16


In [ ]:
print( (T1*T1.H-identity_matrix(4)).n().norm() )
print( ((S1.n()*T1.n())**3 - S1.n()**2).n().norm() )
print( S1.is_symmetric() )
print( (S1*S1.H-identity_matrix(4)).n().norm() )

0.0
1.0819089225884229e-15
True
5.551115123125783e-17


In [55]:
print( (T_dsp6*T_dsp6.H-identity_matrix(40)).n().norm() )
print( ((S_dsp6.n()*T_dsp6.n())**3 - S_dsp6.n()**2).n().norm() )
print( S_dsp6.is_symmetric() )
print( (S_dsp6*S_dsp6.H-identity_matrix(40)).n().norm() )

0.0
1.3336624236470513e-15
True
3.618476623959175e-16


In [6]:
dimg = 2*3**2+3
dimg/(4+1)

21/5

In [22]:
dimg = 2*3**2+3
# Infinite Classes
all_dim_g = [r**2+2*r, 2r**2+r, 2*r**2+r, 2*r**2-r]
all_dcox = [r+1, 2*r-1, r+1, 2*r-2]
alg = ['A', 'B', 'C', 'D']
for rank in range(1,500):
    for level in range(1,500):
        for i in range(len(all_dim_g)):
            dg = all_dim_g[i].subs(r=rank)
            g = all_dcox[i].subs(r=rank)
            if g>=0 and dg>=0 and 7/5==level*dg/(level+g):
                print('Algebra:',alg[i]+str(rank))
                print('level:',level)
                print('')

In [9]:
var('r')
target = 21/5
trank = 3
tdg = 21

# Infinite Classes
all_dim_g = [r**2+2*r, 2r**2+r, 2*r**2+r, 2*r**2-r]
all_dcox = [r+1, 2*r-1, r+1, 2*r-2]
alg = ['A', 'B', 'C', 'D']

for rank in range(0,100):
    for x in range(100):
        for i in range(len(all_dim_g)):
            dg = all_dim_g[i].subs(r=rank)
            g = all_dcox[i].subs(r=rank)
            if g>=0 and (x+g)>0 and target==x*dg/(x+g):
                print('Algebra:',alg[i]+str(rank))
                print('Embedding Index:',x)
                print('')

# Exceptional Cases
dimgs = [248, 133, 78, 52, 14]
coxnums = [30, 18, 12, 9, 4]
alg = ['E8','E7','E6','F4','G2'] 
for i in range(5):
    dg = dimgs[i]
    g = coxnums[i]
    for x in range(100):
        if target==x*dg/(x+g):
            print('Algebra:',alg[i])
            print('Embedding Index:',x)
            print('')


Algebra: B2
Embedding Index: 7

Algebra: C3
Embedding Index: 1



In [ ]:
cartan_types = [["D",4],["A",7],["D",7],["B",38],["E",7],["G",2]]
embedding_indices = [2,2,1,1,15,1,4]

In [ ]:
import collections
from sage.all import CartanType, RootSystem, QQ, ZZ, PuiseuxSeriesRing, crystals, O


def affine_char_list(cartan_type, level, max_ord):
    """
    Computes the specialized characters of an affine Lie algebra at a given level
    as a list of Puiseux series up to O(q^max_ord) relative to the leading term.
    """
    # 1. Normalize and resolve the Cartan types
    ct = CartanType(cartan_type)
    if ct.is_affine():
        ct_affine = ct
        ct_finite = ct.classical()
    else:
        ct_finite = ct
        ct_affine = ct.affine()
        
    # 2. Initialize the Puiseux Series Ring over Integers
    R = PuiseuxSeriesRing(ZZ, 'q')
    q = R.gen()
    
    # 3. Compute central charge constants using the core root system
    h_check = ct_finite.dual_coxeter_number()
    # dim(g) = rank(g) + number of roots (completely bypasses LieAlgebra factory errors)
    dim_g = ct_finite.rank() + len(list(RootSystem(ct_finite).root_lattice().roots()))
    c = QQ(level * dim_g) / QQ(level + h_check)
    
    # 4. Set up ambient space to compute precise inner products for h
    L_finite = RootSystem(ct_finite).ambient_space()
    fw_finite = L_finite.fundamental_weights()
    rho_finite = L_finite.rho()
    
    # 5. Find all dominant highest weights of the given level using the dual null root
    indices = list(ct_affine.index_set())
    
    # Comarks of ct_affine equal the marks of its dual affine type
    # We find them by looking at the null root (delta) of the dual root lattice
    Q_dual = RootSystem(ct_affine.dual()).root_lattice()
    delta_dual = Q_dual.null_root()
    colabels = {i: delta_dual.coefficient(i) for i in indices}

    dominant_weights_coeffs = []
    
    def find_weights(idx, current_sum, current_dict):
        if idx == len(indices):
            if current_sum == level:
                dominant_weights_coeffs.append(current_dict.copy())
            return
        i = indices[idx]
        colab = colabels[i]
        max_val = (level - current_sum) // colab
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * colab, current_dict)
            del current_dict[i]
            
    find_weights(0, 0, {})
    
    # 6. Set up the affine weight lattice to build crystals
    P_affine = RootSystem(ct_affine).weight_lattice()
    Lambda_affine = P_affine.fundamental_weights()
    
    characters = []
    
    # 7. Traverse the crystal for each valid highest weight
    finite_indices = list(ct_finite.index_set())
    
    for coeffs in dominant_weights_coeffs:
        # Accumulate the finite weight projection manually to avoid integer-zero addition errors
        bar_Lambda = fw_finite[finite_indices[0]] * coeffs.get(finite_indices[0], 0)
        for i in finite_indices[1:]:
            bar_Lambda += fw_finite[i] * coeffs.get(i, 0)
            
        # Conformal weight h = (Lambda, Lambda + 2*rho) / (2 * (level + h^V))
        inner_prod = bar_Lambda.inner_product(bar_Lambda + 2 * rho_finite)
        h = QQ(inner_prod) / QQ(2 * (level + h_check))
        
        # Leading modular exponent: h - c/24
        leading_exponent = h - c / QQ(24)
        
        # Accumulate the affine weight vector manually
        weight = Lambda_affine[indices[0]] * coeffs[indices[0]]
        for i in indices[1:]:
            weight += Lambda_affine[i] * coeffs[i]
            
        # Instantiate the highest weight crystal module
        B = crystals.HighestWeight(weight)
        b0 = B.highest_weight_vector()
        
        # Graded dimension tracker via BFS
        counts = [0] * (max_ord + 1)
        queue = collections.deque([(b0, 0)])
        visited = {b0}
        
        while queue:
            b, c0 = queue.popleft()
            counts[c0] += 1
            
            for i in indices:
                fb = b.f(i)
                if fb is not None:
                    # Every application of f_0 lowers the weight by alpha_0, increasing the energy grading by 1
                    new_c0 = c0 + (1 if i == 0 else 0)
                    if new_c0 <= max_ord and fb not in visited:
                        visited.add(fb)
                        queue.append((fb, new_c0))
        
        # 8. Assemble the Puiseux series
        series_sum = sum(counts[depth] * q**depth for depth in range(max_ord + 1))
        char_series = q**leading_exponent * series_sum + O(q**(leading_exponent + max_ord))
        characters.append(char_series)
        
    return characters

In [ ]:

from collections import defaultdict
def affine_char_list(cartan_type, level, max_ord=15):
    '''
    Input: a cartan type (e.g. "A4"), a level, and the maximum order to compute the characters to
    Output: the list of normalised specialised characters of the affine lie algebra
    '''
    # Get the Cartan type
    ct = CartanType(cartan_type)
    if not ct.is_affine():
        ct = ct.affine()
    # Indices
    index = list( ct.index_set() )
    # Affine root system
    WL = RootSystem(ct).weight_lattice(extended=True)
    # Simple roots
    simple_roots = WL.simple_roots()
    # Simple coroots
    simple_coroots = WL.simple_coroots()
    # List of fundamental weights
    Lambda = WL.fundamental_weights()
    # The rank
    rnk = len(index)-1
    # The null root delta
    delta = WL.null_root()
    # The (classical) Cartan matrix
    cartan_mat = CartanMatrix(ct.classical())
    inv_cartan_mat = cartan_mat.inverse()
    # Highest root
    highest_root = WL.classical().highest_root()
    # The quadratic form 
    quadF = matrix(QQ, [[inv_cartan_mat[i,j]*simple_roots[i+1].norm_squared()/highest_root.norm_squared() for i in range(rnk)] for j in range(rnk)] )
    max_F = max(quadF.coefficients())
    
    # Now we find the (classical) weight, root, and coroot lattices as sagemath free module objects over the integers
    weight_lattice = span([vector(ZZ,rnk,{i:1}) for i in range(rnk)],ZZ)
    root_lattice = span(cartan_mat.columns(),ZZ)
    coroot_lattice = span([cartan_mat.columns()[i]*highest_root.norm_squared()/simple_roots[i+1].norm_squared() for i in range(rnk)],ZZ)

    # The power series ring 
    P = PuiseuxSeriesRing(ZZ,'q')
    q = P.gen()

    # Define the generalised theta functions
    def Theta(lamb):
        r'''
        Generalised theta function
        Input: a weight
        Output: the specialised generalised theta function associated to that weight
        '''
        vec_lamb = lamb.to_classical().to_vector()
        dictseries = defaultdict(int)
        qF_gcd = 1/gcd(quadF.coefficients())
        e = 2*level*qF_gcd
        for cr in coroot_lattice:
            vec = level*cr+vec_lamb
            exponent = qF_gcd*vec*quadF*vec
            if exponent <= e*max_ord:
                dictseries[exponent] += 1
            elif exponent > 2*e*max_F*max_ord:
                break
        
        return P(dictseries,e=e).add_bigoh(max_ord)
  

    # Get the affine Weyl group and isolate the classical generators
    affine_weyl = WL.weyl_group()
    classical_nodes = [i for i in index if i != 0]
    weyl_gen = [affine_weyl.simple_reflection(i) for i in classical_nodes]

    # Comarks of ct_affine equal the marks of its dual affine type
    # We find them by looking at the null root (delta) of the dual root lattice
    Q_dual = RootSystem(ct.dual()).root_lattice()
    delta_dual = Q_dual.null_root()
    comarks = {i: delta_dual.coefficient(i) for i in index}

    # Now we find all the integrable weights
    int_weights_coeffs = []
    def find_weights(idx, current_sum, current_dict):
        if idx == rnk+1:
            if current_sum == level:
                int_weights_coeffs.append(current_dict.copy())
            return
        i = index[idx]
        max_val = (level - current_sum) // comarks[i] 
        for val in range(max_val + 1):
            current_dict[i] = val
            find_weights(idx + 1, current_sum + val * comarks[i], current_dict)
            del current_dict[i]
            
    find_weights(0, 0, {})
    # Number of integrable highest weight representations
    num_reps = len(int_weights_coeffs)
    # The integrable weights
    int_weights = [ sum(Lambda[i]*int_weights_coeffs[j][i] for i in index) for j in range(num_reps)] 
    def are_equivalent(w1, w2):
        '''Function to determine if two weights are related via shift by a coroot'''

        c1 = vector(QQ, [w1.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        c2 = vector(QQ, [w2.monomial_coefficients().get(i+1,0) for i in range(rnk)] )
        return (c1-c2)/level in coroot_lattice #and w1.monomial_coefficients().get(0,0)==w2.monomial_coefficients().get(0,0)

    # Now loop over these weights and calculate the character for each
    char_list = []
    for iwght in int_weights:
        # The corresponding integrable representation
        int_rep = IntegrableRepresentation(iwght)
        # First let's calculate the set of maximal dominant weights
        md_weights = int_rep.dominant_maximal_weights()
        # Now we find a full set of representatives up to translations by the coroot lattice
        full_maximal_representatives = list( md_weights )
        queue = list( md_weights )
        while queue:
            current = queue.pop(0)
            for g in weyl_gen:
                nxt = g.action(current)
                # Only keep nxt if it represents a brand new class modulo the coroots
                if not any(are_equivalent(nxt, r) for r in full_maximal_representatives):
                    full_maximal_representatives.append(nxt)
                    queue.append(nxt)
        
        # Now compute the character by summing over these weights
        char = P(0).add_bigoh(max_ord)
        for wght in full_maximal_representatives:
            char += q**(int_rep.modular_characteristic(wght))*P( int_rep.string(wght, depth=max_ord) )*Theta(wght)
        # Update the list of characters
        char_list.append(char)
        
    return char_list


In [ ]:
''' Code to compute the modular matrices '''
xe = embedding_indices[0]
ct = cartan_types[0]
C3_2_char_list = affine_char_list(['C',3,1],2,6)
all_char_decomps = []
for ct in cartan_types:
    if ct == ['B',38]:
        continue
    emb_char_list = affine_char_list(ct+[1],1,6)
    all_char_decomps.append( char_decomp(emb_char_list,C3_2_char_list,5) )

KeyboardInterrupt: 